# FormulaLM — PROMPT 8: QLoRA Fine-Tuning (Qwen3-1.7B, Non-Thinking, Unsloth)

Ready-to-run training notebook for a **free Google Colab or Kaggle T4 GPU (16 GB VRAM)**.
This notebook wraps [`scripts/train_qlora.py`](../scripts/train_qlora.py) and [`config/train_qlora.yaml`](../config/train_qlora.yaml) so CLI and notebook runs are 100% identical.

## Hyperparameters (one-sentence explanation each)

| Hyperparameter | Value | Why |
|---|---|---|
| `base_id` + `base_revision` | `Qwen/Qwen3-1.7B` (`70d244cc...`) | Small Apache-2.0 base model that fits easily in 4-bit on T4 and runs fast on a CPU laptop after GGUF quantization. |
| `load_in_4bit` | `True` (NF4 QLoRA) | Quantizes the frozen base weights to 4-bit during training so the 1.7B model + activations fit comfortably in 16 GB VRAM. |
| `enable_thinking` | `False` | Persists Qwen3's non-thinking mode so the model outputs only the `=...` formula without `<think>` chatter. |
| `train_on_responses_only` | `True` (`<\|im_start\|>assistant\n<think>\n\n</think>\n\n`) | Masks system, schema, and request tokens (`-100`) so 100% of training loss focuses on generating the Excel formula accurately. |
| `max_seq_length` | `512` | Longest example across all 9,000 rows is 603 chars (~251 tokens), so 512 gives >2x headroom with zero truncation while saving VRAM. |
| `lora_r` | `16` | Rank-16 low-rank adapters give enough capacity to learn all 36 Excel functions and column mapping without overfitting. |
| `lora_alpha` | `32` | Standard `2 * r` scaling factor that keeps adapter update magnitudes well-conditioned at `lr=2e-4`. |
| `lora_dropout` | `0.0` | Zero dropout unlocks Unsloth's fused fast LoRA kernels, and 8,000 diverse examples already regularize well. |
| `target_modules` | `q,k,v,o,gate,up,down_proj` | Adapting all 7 attention and MLP linear layers consistently beats attention-only LoRA on syntax-sensitive code/formula tasks. |
| `per_device_train_batch_size` × `grad_accum` | `4 × 4 = 16` | Effective batch size of 16 gives smooth gradients on T4 16 GB VRAM without OOM (500 optimizer steps per epoch). |
| `num_train_epochs` | `2` (or `3`) | Two passes over 8,000 examples (1,000 steps, ~15 min on T4) converge cleanly without memorizing synthetic templates; bump to 3 only if val loss is still falling. |
| `learning_rate` | `2e-4` | Standard QLoRA learning rate for small 1.7B adapters (higher than full fine-tuning because only ~1% of parameters train). |
| `lr_scheduler_type` + `warmup_ratio` | `cosine` + `0.05` | Warms up over the first 5% of steps to stabilize early adapter updates, then smoothly decays LR to zero. |
| `optim` | `adamw_8bit` | Cuts optimizer state VRAM roughly in half with negligible quality difference on T4. |
| `eval_steps` / `save_steps` | `100` / `100` | Evaluates on all 500 validation examples and saves a resumable checkpoint every 100 steps (keeping best by `eval_loss`). |
| `seed` | `42` | Fixes weight init, data shuffling, and evaluation sampling for reproducible runs. |

In [ ]:
# 1. Check GPU (must be NVIDIA T4 / CUDA GPU, NOT TPU or CPU)
import torch

assert torch.cuda.is_available(), 'No CUDA GPU detected! Switch runtime to T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0), '| VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 2. Environment Setup (Run **2A for Google Colab** OR **2B for Kaggle**)

In [ ]:
# 2A. GOOGLE COLAB ONLY — Install verified Unsloth Qwen3 stack & optionally mount Google Drive
# (Verified pins from junaiddshaukat/finetune-anywhere, checked Nov 2026)
MOUNT_DRIVE = True
if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print('Drive mount skipped (normal in VS Code/IDE Colab extension):', exc)

%pip install -q sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer pyyaml formulas openpyxl pandas
%pip install -q --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
!unsloth install-kernels
%pip install -q --no-deps --upgrade "torchao>=0.16.0"
%pip install -q transformers==4.57.6
%pip install -q --no-deps trl==0.22.2

### 2B. Kaggle-Specific Setup (skip this cell if you are on Google Colab)
- Enable **Accelerator: GPU T4 x2 (or P100)** and **Internet: On** in the right-hand Kaggle Settings panel.
- Checkpoints and adapters save under `/kaggle/working/formulalm_outputs/` (retained across Save Version runs; session cap is 12 h, so `--resume auto` recovers cleanly if you attach a prior output dataset).
- Add `HF_TOKEN` and (while repo is private) `GITHUB_TOKEN` in **Add-ons → Secrets**.

In [ ]:
# 2B. KAGGLE ONLY — Run this cell instead of 2A when on Kaggle
RUN_KAGGLE_SETUP = False
if RUN_KAGGLE_SETUP:
    import os
    try:
        from kaggle_secrets import UserSecretsClient
        _ks = UserSecretsClient()
        for _key in ('GITHUB_TOKEN', 'HF_TOKEN'):
            try:
                os.environ[_key] = _ks.get_secret(_key)
                print(f'Loaded Kaggle secret: {_key}')
            except Exception:
                pass
    except Exception as exc:
        print('kaggle_secrets unavailable:', exc)
    %pip install -q torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
    %pip install -q unsloth pyyaml formulas openpyxl pandas
    %pip install -q --no-deps --upgrade "torchao>=0.16.0"
    %pip install -q transformers==4.57.6
    %pip install -q --no-deps trl==0.22.2

In [ ]:
# 3. Clone / pull FormulaLM repo and regenerate verified train/val/test_synthetic splits (seed=42)
import getpass, os, pathlib, subprocess, sys

IS_KAGGLE = pathlib.Path('/kaggle/working').is_dir()
REPO_DIR = pathlib.Path('/kaggle/working/FormulaLM' if IS_KAGGLE else '/content/FormulaLM')

if not REPO_DIR.exists():
    token = os.environ.get('GITHUB_TOKEN')
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')
        except Exception:
            token = None
    if not token:
        token = getpass.getpass('GitHub PAT for abubakarp789/FormulaLM (hidden, empty if public): ').strip()
    url = f'https://{token}@github.com/abubakarp789/FormulaLM.git' if token else 'https://github.com/abubakarp789/FormulaLM.git'
    subprocess.run(['git', 'clone', url, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

%cd {REPO_DIR}
if str(REPO_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_DIR / 'src'))

!python -m src.data.build_dataset --workers 2

## 4. Quick Smoke Test (200 train / 50 val examples, ~1–2 min)
Always run this first to verify 4-bit loading, non-thinking chat template, response-only loss masking, checkpointing, and loss decrease before launching the full 8,000-example run.

In [ ]:
# 4. Run 200-example smoke test (~1-2 min on T4)
!python scripts/train_qlora.py --smoke --run-name qlora_v1

print('=== Smoke test loss_log.csv ===')
!cat results/train_qlora_v1_smoke/loss_log.csv

## 5. Full QLoRA Training Run (8,000 train / 500 val, 2 epochs, auto-resume)
- Checkpoints save every **100 steps** to Drive (`/content/drive/MyDrive/FormulaLM/qlora_v1/checkpoints`) or Kaggle output (`/kaggle/working/formulalm_outputs/qlora_v1/checkpoints`).
- If your Colab/Kaggle session hits a time limit or disconnects, re-run Cells 1–3 and **re-run Cell 5** — `--resume auto` picks up from the highest saved `checkpoint-*` automatically.

In [ ]:
# 5. Full QLoRA fine-tuning (8,000 train / 500 val, 2 epochs = 1,000 steps, ~15 min on T4)
import pathlib

RUN_NAME = 'qlora_v1'
EPOCHS = 2  # Change to 3 if val_loss is still dropping strongly at step 1000

if pathlib.Path('/kaggle/working').is_dir():
    OUTPUT_DIR = f'/kaggle/working/formulalm_outputs/{RUN_NAME}'
elif pathlib.Path('/content/drive/MyDrive').is_dir():
    OUTPUT_DIR = f'/content/drive/MyDrive/FormulaLM/{RUN_NAME}'
else:
    OUTPUT_DIR = f'outputs/{RUN_NAME}'

print('Saving checkpoints & adapter to:', OUTPUT_DIR)
!python scripts/train_qlora.py \
  --run-name {RUN_NAME} \
  --output-dir {OUTPUT_DIR} \
  --epochs {EPOCHS} \
  --resume auto

In [ ]:
# 6. Inspect training/validation loss log & run summary (paste this output back to review!)
import json, pandas as pd

res_dir = pathlib.Path(f'results/train_{RUN_NAME}')
print('=== run_summary.json ===')
print((res_dir / 'run_summary.json').read_text())

print('=== loss_log.csv ===')
df_loss = pd.read_csv(res_dir / 'loss_log.csv')
print(df_loss.to_string(index=False))

In [ ]:
# 7. Optional: Push final LoRA adapter to Hugging Face Hub + push lightweight training logs to GitHub
# Secrets stay in env vars / Colab Secrets / Kaggle Secrets — never committed.
PUSH_ADAPTER_TO_HF = False
HF_REPO_ID = 'abubakar56/formulalm-qwen3-1.7b-lora'

if PUSH_ADAPTER_TO_HF:
    import getpass, os
    from huggingface_hub import HfApi
    hf_token = os.environ.get('HF_TOKEN')
    if not hf_token:
        try:
            from google.colab import userdata
            hf_token = userdata.get('HF_TOKEN')
        except Exception:
            hf_token = getpass.getpass('Hugging Face write token (hidden): ').strip()
    api = HfApi(token=hf_token)
    api.create_repo(HF_REPO_ID, exist_ok=True, repo_type='model')
    api.upload_folder(folder_path=f'{OUTPUT_DIR}/adapter', repo_id=HF_REPO_ID, repo_type='model')
    print('Uploaded LoRA adapter to https://huggingface.co/' + HF_REPO_ID)

# Push ONLY lightweight CSV/JSON logs in results/train_* back to GitHub (never model weights)
PUSH_LOGS_TO_GITHUB = False
if PUSH_LOGS_TO_GITHUB:
    !git config user.name "abubakarp789"
    !git config user.email "abubakarp789@gmail.com"
    !git add results/train_*
    !git commit -m "results: add QLoRA v1 training and validation loss logs"
    !git pull --rebase origin main
    !git push origin main